**Конвертация таблицы из docx в csv**

In [2]:
!pip install python-docx

In [3]:
# Загрузка библиотек
#from docx import Document
import pandas as pd
import numpy as np
import os
import docx
import re

In [4]:
path = 'E:/IrkutskProject/Data/'
years = [2016, 2017, 2018, 2019, 2020]
original_names = {
    2016: 'soc_znach2016.docx',
    2017: 'soc_znach2017.docx',
    2018: 'soc_znach2018.docx',
    2019: 'soc_znach2019.docx',
    2020: 'soc_znach2020.docx'
}
original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/soc_znach2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/soc_znach2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/soc_znach2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/soc_znach2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/soc_znach2020.docx найден по указанному пути


In [5]:
def extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    
    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        
        # Шаг 1: Ищем текст заголовка
        # Проверяем только абзацы ('p'), чтобы не искать внутри других таблиц случайно
        if not target_found and element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")
                # continue 
        
        # Шаг 2: После того как текст найден, ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем, сколько таблиц встретилось в документе до текущего момента
            # Это и будет порядковый индекс в doc.tables
            table_index = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            
            print(f"Таблица найдена (индекс в документе: {table_index})")
            return doc.tables[table_index]

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    else:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

def table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)
    
    # Создаем DataFrame
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

In [6]:
try:
    df = {}
    target_block = "Заболеваемость и контингенты пациентов активным туберкулезом\nпо субъектам Российской Федерации"
    for year in years:
        display(year)
        table = extract_tuberculosis_table(original_url[year], target_block)
        df[year] = table_to_dataframe(table)
        df[year].to_csv(f'E:/IrkutskProject/Data/{year}/tuberculosis_data.csv', index=False, encoding='utf-8-sig')
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

2016

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 4)


2017

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2018

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2019

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2020

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


In [7]:
df[2016]

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения
1,СУБЪЕКТЫ ФЕДЕРАЦИИ,2015,2016,2015,2016,2015,2016,2015,2016
2,Российская Федерация,84515,78121,"57,7","53,3",189186,178080,"129,1","121,5"
3,Центральный федеральный округ,14700,13375,"37,7","34,2",26988,23615,"69,0","60,4"
4,Белгородская область,420,333,"27,1","21,5",564,428,"36,4","27,6"
...,...,...,...,...,...,...,...,...,...
92,Магаданская область,109,82,"74,0","56,0",228,174,"155,8","118,9"
93,Сахалинская область,320,306,"65,6","62,8",1177,1109,"241,5","227,6"
94,Еврейская автономная область,209,202,"125,0","121,6",592,500,"356,4","301,0"
95,Чукотский автономный округ,79,86,"156,9","171,5",168,187,"334,9","372,8"


In [8]:
def get_tuberculosis_dataframe (df, years, clarified_flag):
    columns = [
        'Субъект',
        'Показатель',
        'Уточнение',
        'Год',
        'Значение',
        'Источник'
    ]

    data = []

    for year in years:
        for row_number in range(2, df[year].shape[0]):
            # число пациентов с впервые в жизни установ.диагнозом + абсолютные показатели
            new_row = []
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[2]) # Показатель
            new_row.append(df[year].iloc[0, 2]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 1]) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 2]) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов с впервые в жизни установ.диагнозом + на 100000 соот.населения
            new_row = []
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[4]) # Показатель
            new_row.append(df[year].iloc[0, 4]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 3]) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 4]) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов состоящих под диспансерным наблюдением на конец года + абсолютные числа
            new_row = []
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[6]) # Показатель
            new_row.append(df[year].iloc[0, 6]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 5]) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 6]) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов состоящих под диспансерным наблюдением на конец года + на 100000 соот.населения
            new_row = []
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[8]) # Показатель
            new_row.append(df[year].iloc[0, 8]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 7]) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 8]) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

    df_long = pd.DataFrame(data, columns=columns)
    return df_long

In [9]:
df_tuberculosis_long = get_tuberculosis_dataframe (df, years, False)
df_tuberculosis_long.to_csv(f'{path}Common/tuberculosis_2016-2020.csv', index=False)

In [10]:
df_tuberculosis_long_clarified = get_tuberculosis_dataframe (df, years, True)
df_tuberculosis_long_clarified.to_csv(f'{path}Common/tuberculosis_clarified_2016-2020.csv', index=False)

In [11]:
years = [2016, 2017, 2018]
df_tuberculosis_long = get_tuberculosis_dataframe (df, years, False)
df_tuberculosis_long.to_csv(f'{path}Common_2016-2018/tuberculosis_2016-2018.csv', index=False)

df_tuberculosis_long_clarified = get_tuberculosis_dataframe (df, years, True)
df_tuberculosis_long_clarified.to_csv(f'{path}Common_2016-2018/tuberculosis_clarified_2016-2018.csv', index=False)